# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step example of loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
This dataset is described by a Croissant schema and can be accessed from the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install `mlcroissant` if not already available
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Let's list the available record sets and their associated fields using their `@id`. Here we use the Croissant metadata schema.

In [ ]:
# List all record sets and their fields by @id

record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}")

for rs in record_sets:
    print(f"\nRecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '(none)')}")
    print(f"  Description: {rs.get('description', '(none)')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for f in fields:
        if isinstance(f, dict):
            print(f"    - @id: {f.get('@id', None)} | Name: {f.get('name','')} | DataType: {f.get('dataType','')}")
        elif isinstance(f, str):
            print(f"    - @id: {f}")

## 3. Data Extraction
We'll extract data from each record set (using its `@id`) to a pandas DataFrame.

Refer to the record set `@id`s printed above. Below, we'll load all available record sets into named DataFrames using their `@id`.

In [ ]:
# Collect all recordSet @ids for data extraction
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}

# Iterate through each recordSet @id and load the records into a DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from RecordSet '{record_set_id}'")
    else:
        print(f"No records found for RecordSet '{record_set_id}'")

# Display columns of the first available DataFrame (if any)
if dataframes:
    first_rs_id = next(iter(dataframes))
    print(f"Available columns in DataFrame for RecordSet {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)

Let's perform some basic processing steps, such as filtering, normalizing a numeric field (e.g., 'Age'), and grouping by a categorical field (e.g., 'Sex' if available).

Replace the following variable values with actual `@id`s or column names as shown in your RecordSet overview above.

In [ ]:
# Example: EDA on the first available record set
if dataframes:
    record_set_id = first_rs_id  # Use the previously discovered first record set
    df = dataframes[record_set_id]

    # Choose a likely numeric field, e.g., 'Age' (use the right column name or @id)
    potential_numeric_fields = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or df[col].dtype.kind in 'if']
    if potential_numeric_fields:
        numeric_field = potential_numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")

        # Set a threshold for demonstration (e.g., age > 40)
        try:
            th = float(df[numeric_field].dropna().mean())
            threshold = th if th > 0 else 10
        except Exception:
            threshold = 10

        # Filter records
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field
        mean = filtered_df[numeric_field].mean()
        std = filtered_df[numeric_field].std()
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    else:
        print("No suitable numeric field found for EDA.")

    # Try grouping by a categorical field, e.g. 'Sex', if available
    potential_group_fields = [col for col in df.columns if 'sex' in col.lower() or 'group' in col.lower() or df[col].dtype == object]
    if filtered_df is not None and potential_group_fields:
        group_field = potential_group_fields[0]
        if group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean_'+numeric_field)
            print(f"Grouped data by '{group_field}' (mean {numeric_field}):")
            display(grouped_df)
else:
    print("No DataFrame to analyze.")

## 5. Visualization

Let's visualize the distribution of a numeric field (e.g., 'Age') and its relationship to a group/categorical variable (e.g., 'Sex').

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'filtered_df' in locals() and not filtered_df.empty and 'numeric_field' in locals():
    plt.figure(figsize=(7,5))
    sns.histplot(filtered_df[numeric_field].dropna(), kde=True, color='dodgerblue')
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if 'group_field' in locals() and group_field in filtered_df.columns:
        plt.figure(figsize=(7,5))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to:
- Load and inspect metadata and record structure for the FAIR² dataset,
- Extract records by their `@id`,
- Perform exploratory analysis on a numeric variable (e.g., filtering, normalization),
- Group data by a key attribute,
- Visualize feature distributions.

Next steps may include deeper feature engineering, predictive modeling, or domain-specific statistical analysis depending on research questions.